In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()

EVALUATION_NAME = "chatbs-base"
CONFIG_FILENAME = "config.fullcontext.yaml"

PREDICTION_FILES = {
    "fullcontext": "evaluations/chatbs-base/explainer/fullcontext/exp__20260420220709/RESULTS.jsonl",
    "grasp": "evaluations/chatbs-base/explainer/grasp/exp_202604201325/RESULTS.jsonl",
    "llmbased": "evaluations/chatbs-base/explainer/llmbased/exp__20260420235310/RESULTS.jsonl",
    "ours":"evaluations/chatbs-base/explainer/results/exp__20260419212418/RESULTS.jsonl"
}

ENABLE_LLM_JUDGE = False
JUDGE_LLM = {
    "llm_type": "openai",
    "llm_library": "dspy",
    "llm_config": {
        "model": "gpt-4o-mini",
        "temperature": 0,
        "max_tokens": 300,
    },
}

ENABLED_METRICS = [
    "answer_token_overlap",
    "ground_truth_entity_coverage",
    "llm_answer_quality",
]

MAX_EXAMPLES_PER_RUN = None

SAVE_DIR = REPO_ROOT / "evaluations" / EVALUATION_NAME / "analysis"
SAVE_PER_EXAMPLE_CSV = None  # e.g. SAVE_DIR / "per_example_metrics.csv"
SAVE_SUMMARY_CSV = None      # e.g. SAVE_DIR / "run_summary.csv"


# Evaluation Notebook

Edit the first cell only for normal use:

- Set `EVALUATION_NAME` and `CONFIG_FILENAME` to choose the evaluation folder.
- Add one or more `codename -> RESULTS.jsonl` entries to `PREDICTION_FILES`.
- Turn on `ENABLE_LLM_JUDGE` if you want the LLM-based grading metric.
- Add new metric functions in the metric cell and register them in `METRIC_REGISTRY`.


In [29]:
import asyncio
import json
import re
from collections import Counter
from collections.abc import Callable, Mapping
from pathlib import Path
from typing import Any, Dict, List

import dspy
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from pydantic import BaseModel, Field

from src.config.experiment import FullContextExperimentConfig
from src.experiment.ground_truth import GTInfo
from src.llm import LLM
from src.utils.utils import load_config

load_dotenv(REPO_ROOT / ".env")


True

In [30]:
def resolve_repo_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    if path.is_absolute():
        return path
    return (REPO_ROOT / path).resolve()


def strip_citations(text: Any) -> str:
    cleaned = re.sub(
        r"<cite,\s*id=\d+>.*?</cite>",
        " ",
        str(text or ""),
        flags=re.IGNORECASE,
    )
    return re.sub(r"\s+", " ", cleaned).strip()


def normalize_text(text: Any) -> str:
    text = strip_citations(text)
    text = re.sub(r"[^a-zA-Z0-9]+", " ", text.lower())
    return re.sub(r"\s+", " ", text).strip()


def normalize_question(text: Any) -> str:
    return normalize_text(text)


def unique_preserving_order(items: list[str]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for item in items:
        if not item or item in seen:
            continue
        seen.add(item)
        ordered.append(item)
    return ordered


def load_ground_truth_bundle(evaluation_name: str, config_filename: str) -> dict[str, Any]:
    evaluation_dir = resolve_repo_path(Path("evaluations") / evaluation_name)
    config_path = evaluation_dir / config_filename
    if not config_path.exists():
        raise FileNotFoundError(f"Config file not found: {config_path}")

    raw_config = load_config(str(config_path))
    config = FullContextExperimentConfig.model_validate(raw_config)
    gt_path = resolve_repo_path(config.gt.save_loc)
    ground_truth = GTInfo(str(gt_path))
    records = [item.model_dump() for item in ground_truth.gt_info]

    by_id = {record["id"]: record for record in records}
    by_question = {
        normalize_question(record["question"]): record
        for record in records
    }

    return {
        "evaluation_dir": evaluation_dir,
        "config_path": config_path,
        "config": config,
        "ground_truth_path": gt_path,
        "records": records,
        "by_id": by_id,
        "by_question": by_question,
    }


GT_BUNDLE = load_ground_truth_bundle(EVALUATION_NAME, CONFIG_FILENAME)
GROUND_TRUTH_RECORDS = GT_BUNDLE["records"]
GROUND_TRUTH_BY_ID = GT_BUNDLE["by_id"]
GROUND_TRUTH_BY_QUESTION = GT_BUNDLE["by_question"]

print(f"Config: {GT_BUNDLE['config_path']}")
print(f"Ground truth: {GT_BUNDLE['ground_truth_path']}")
print(f"Ground-truth examples: {len(GROUND_TRUTH_RECORDS)}")

display(
    pd.DataFrame(GROUND_TRUTH_RECORDS)[["id", "question", "qtype"]].head()
)


Config: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/config.fullcontext.yaml
Ground truth: /home/desild/work/research/LLM-Workflow-Explorer/evaluations/chatbs-base/ground_truth/ground_truth_data.json
Ground-truth examples: 103


,id,question,qtype
0,gt_0,"\nHow many unique ""experiment execution"" are t...","[single, numeric, ISP]"
1,gt_1,\nIn what places do we utilize AI in this work...,"[multi, entity, SI]"
2,gt_2,\nWhat are the extracted KG entities for the e...,"[multi, literal, ISP]"
3,gt_3,\nwhy did some ingredients have missing values...,"[multi, literal, ISP]"
4,gt_4,\nwhat are the instructions used by the LLM to...,"[multi, entity, literal, SI]"


In [31]:
def grasp_input_config(record:Dict[str, Any]) -> Dict[str, Any]:
    return record


INPUT_AUGMENTATION_MAP = {
    "grasp":grasp_input_config
}


In [35]:
import dycomutils as common_utils
import os

def read_jsonl(path_value: str | Path) -> list[dict[str, Any]]:
    path = resolve_repo_path(path_value)
    if not path.exists():
        raise FileNotFoundError(f"Prediction file not found: {path}")

    records: list[dict[str, Any]] = []
    
    for line_number, record in enumerate(common_utils.serialization.load_jsonl(path), start=1):
        record["_prediction_path"] = str(path)
        record["_line_number"] = line_number
        records.append(record)

    if len(records)>1:
        print(f"{str(path).split(os.sep)[-3]}\n{sorted(records[0].keys())}\n")
    return records


def load_prediction_runs(
    prediction_files: Mapping[str, str | Path],
    data_augmentations: dict[str, Callable]
    ) -> dict[str, list[dict[str, Any]]]:
    if not prediction_files:
        raise ValueError("Add at least one entry to PREDICTION_FILES in the first cell.")

    runs: dict[str, list[dict[str, Any]]] = {}
    for codename, file_path in prediction_files.items():
        records = read_jsonl(file_path)
        if MAX_EXAMPLES_PER_RUN is not None:
            records = records[:MAX_EXAMPLES_PER_RUN]
            
        runs[codename] = records
        if codename in data_augmentations:
            for i,r in enumerate(runs[codename]):
                runs[codename][i] = data_augmentations[codename](r)
    return runs


PREDICTION_RUNS = load_prediction_runs(
    PREDICTION_FILES,
    INPUT_AUGMENTATION_MAP
    )

prediction_inventory_df = pd.DataFrame(
    [
        {
            "run": codename,
            "file": records[0]["_prediction_path"] if records else "",
            "rows": len(records),
        }
        for codename, records in PREDICTION_RUNS.items()
    ]
)
display(prediction_inventory_df)


fullcontext
['_line_number', '_prediction_path', 'answer', 'evidence', 'id', 'question', 'relevant_entities', 'time_taken']

grasp
['_line_number', '_prediction_path', 'elapsed', 'error', 'id', 'known', 'messages', 'output', 'task', 'time_taken', 'type']

llmbased
['_line_number', '_prediction_path', 'answer', 'evidence', 'id', 'question', 'relevant_entities', 'time_taken']

results
['_line_number', '_prediction_path', 'answer', 'grounding', 'id', 'intermediary_results', 'judge', 'predecessor_context', 'predecessor_info', 'question', 'report', 'retrieved_objects', 'schema_info_used', 'schema_reasoning', 'step_context', 'synthetic_questions_plan', 'time_taken']



,run,file,rows
0,fullcontext,/home/desild/work/research/LLM-Workflow-Explor...,103
1,grasp,/home/desild/work/research/LLM-Workflow-Explor...,103
2,llmbased,/home/desild/work/research/LLM-Workflow-Explor...,103
3,ours,/home/desild/work/research/LLM-Workflow-Explor...,112


## Metric Hooks

Each metric is just a function with this shape:

`metric(pred: dict, actual: dict) -> dict[str, Any]`

To add a new metric, define another function below and register it in `METRIC_REGISTRY`.


In [ ]:
def tokenize(text: Any) -> list[str]:
    normalized = normalize_text(text)
    if not normalized:
        return []
    return normalized.split()


def extract_ground_truth_entity_values(actual: dict[str, Any]) -> list[str]:
    values: list[str] = []
    for entity in actual.get("entities", []):
        if isinstance(entity, dict):
            for value in entity.values():
                if value is not None and str(value).strip():
                    values.append(str(value))
        elif entity is not None and str(entity).strip():
            values.append(str(entity))
    return unique_preserving_order(values)


def extract_prediction_surface_forms(pred: dict[str, Any]) -> list[str]:
    values: list[str] = []

    answer = strip_citations(pred.get("answer", ""))
    if answer:
        values.append(answer)

    for entity in pred.get("relevant_entities", []):
        if isinstance(entity, dict):
            for key in ("id", "label"):
                value = entity.get(key)
                if value is not None and str(value).strip():
                    values.append(str(value))
            for value in entity.get("types", []) or []:
                if value is not None and str(value).strip():
                    values.append(str(value))
        elif entity is not None and str(entity).strip():
            values.append(str(entity))

    for triple in pred.get("evidence", []):
        if isinstance(triple, dict):
            for key in (
                "subject_id",
                "subject_label",
                "predicate_id",
                "predicate_label",
                "object_id",
                "object_label",
            ):
                value = triple.get(key)
                if value is not None and str(value).strip():
                    values.append(str(value))
        elif triple is not None and str(triple).strip():
            values.append(str(triple))

    return unique_preserving_order(values)


def text_is_covered(target: str, candidates: list[str]) -> bool:
    target_tokens = set(tokenize(target))
    if not target_tokens:
        return False

    for candidate in candidates:
        candidate_tokens = set(tokenize(candidate))
        if not candidate_tokens:
            continue
        if target_tokens <= candidate_tokens:
            return True
    return False



class AnswerQualityScores(BaseModel):
    completeness: float = Field(ge=0.0, le=1.0)
    accuracy: float = Field(ge=0.0, le=1.0)
    relevance: float = Field(ge=0.0, le=1.0)


class AnswerQualityJudgeSignature(dspy.Signature):
    """Score answer completeness, accuracy, and relevance against the ground truth."""

    question: str = dspy.InputField()
    ground_truth_answer: str = dspy.InputField()
    model_answer: str = dspy.InputField()
    completeness: float = dspy.OutputField(desc="How much of the ground truth is covered, from 0 to 1.")
    accuracy: float = dspy.OutputField(desc="How faithful the answer is to the ground truth, from 0 to 1.")
    relevance: float = dspy.OutputField(desc="How directly the answer addresses the question, from 0 to 1.")


_judge_llm: Any | None = None


def get_judge_llm() -> Any:
    global _judge_llm
    if _judge_llm is None:
        _judge_llm = LLM(
            JUDGE_LLM["llm_type"],
            JUDGE_LLM["llm_library"],
            **JUDGE_LLM["llm_config"],
        )
    return _judge_llm


In [ ]:
MetricFn = Callable[[dict[str, Any], dict[str, Any]], dict[str, Any]]


def metric_answer_token_overlap(pred: dict[str, Any], actual: dict[str, Any]) -> dict[str, Any]:
    pred_counter = Counter(tokenize(pred.get("answer", "")))
    actual_counter = Counter(tokenize(actual.get("answer", "")))

    overlap = sum((pred_counter & actual_counter).values())
    pred_total = sum(pred_counter.values())
    actual_total = sum(actual_counter.values())

    precision = overlap / pred_total if pred_total else 0.0
    recall = overlap / actual_total if actual_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "answer_token_precision": precision,
        "answer_token_recall": recall,
        "answer_token_f1": f1,
    }


def metric_ground_truth_entity_coverage(pred: dict[str, Any], actual: dict[str, Any]) -> dict[str, Any]:
    gt_values = extract_ground_truth_entity_values(actual)
    pred_values = extract_prediction_surface_forms(pred)
    covered_values = [value for value in gt_values if text_is_covered(value, pred_values)]
    missing_values = [value for value in gt_values if value not in covered_values]

    coverage = len(covered_values) / len(gt_values) if gt_values else float("nan")
    return {
        "gt_entity_total": len(gt_values),
        "gt_entity_covered": len(covered_values),
        "gt_entity_coverage": coverage,
        "missing_gt_entities": json.dumps(missing_values, ensure_ascii=True),
    }


def build_llm_answer_quality_metric() -> MetricFn:
    judge = dspy.Predict(AnswerQualityJudgeSignature)

    def metric(pred: dict[str, Any], actual: dict[str, Any]) -> dict[str, Any]:
        judge_llm = get_judge_llm()
        with dspy.context(lm=judge_llm.llm):
            scores = judge(
                question=actual.get("question", ""),
                ground_truth_answer=actual.get("answer", ""),
                model_answer=strip_citations(pred.get("answer", "")),
            )

        validated_scores = AnswerQualityScores(
            completeness=float(scores.completeness),
            accuracy=float(scores.accuracy),
            relevance=float(scores.relevance),
        )
        return {
            "llm_completeness": validated_scores.completeness,
            "llm_accuracy": validated_scores.accuracy,
            "llm_relevance": validated_scores.relevance,
        }

    metric.__name__ = "llm_answer_quality"
    return metric

def build_nli_metric(pred: dict[str, Any], actual: dict[str, Any]) -> dict[str, Any]:
    
    return {
        "nli_score":0
        }


METRIC_REGISTRY: dict[str, MetricFn] = {
    "answer_token_overlap": metric_answer_token_overlap,
    "ground_truth_entity_coverage": metric_ground_truth_entity_coverage,
}

if ENABLE_LLM_JUDGE:
    METRIC_REGISTRY["llm_answer_quality"] = build_llm_answer_quality_metric()

ACTIVE_METRIC_NAMES: list[str] = []
ACTIVE_METRICS: list[MetricFn] = []
skipped_metrics: list[str] = []

for metric_name in ENABLED_METRICS:
    metric_fn = METRIC_REGISTRY.get(metric_name)
    if metric_fn is None:
        skipped_metrics.append(metric_name)
        continue
    ACTIVE_METRIC_NAMES.append(metric_name)
    ACTIVE_METRICS.append(metric_fn)

print("Active metrics:", ACTIVE_METRIC_NAMES)
if skipped_metrics:
    print("Skipped metrics:", skipped_metrics)


Active metrics: ['answer_token_overlap', 'ground_truth_entity_coverage']
Skipped metrics: ['llm_answer_quality']


In [7]:
def resolve_ground_truth_record(pred: dict[str, Any]) -> dict[str, Any] | None:
    pred_id = pred.get("id")
    if pred_id in GROUND_TRUTH_BY_ID:
        return GROUND_TRUTH_BY_ID[pred_id]

    question_key = normalize_question(pred.get("question", ""))
    return GROUND_TRUTH_BY_QUESTION.get(question_key)


def evaluate_run(
    run_name: str,
    predictions: list[dict[str, Any]],
    metric_functions: list[MetricFn],
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    for pred in predictions:
        actual = resolve_ground_truth_record(pred)
        row: dict[str, Any] = {
            "run": run_name,
            "prediction_path": pred.get("_prediction_path"),
            "prediction_line": pred.get("_line_number"),
            "prediction_id": pred.get("id"),
            "ground_truth_found": actual is not None,
            "question": pred.get("question", ""),
            "pred_answer": strip_citations(pred.get("answer", "")),
            "prediction_time_taken": pred.get("time_taken"),
        }

        if actual is None:
            row["evaluation_error"] = "ground_truth_not_found"
            rows.append(row)
            continue

        row.update(
            {
                "ground_truth_id": actual.get("id"),
                "ground_truth_qtype": json.dumps(actual.get("qtype", []), ensure_ascii=True),
                "ground_truth_tags": json.dumps(actual.get("tags", {}), ensure_ascii=True, sort_keys=True),
                "ground_truth_answer": actual.get("answer", ""),
            }
        )

        for metric_fn in metric_functions:
            try:
                row.update(metric_fn(pred, actual))
            except Exception as exc:
                row[f"{metric_fn.__name__}_error"] = str(exc)

        rows.append(row)

    return rows


evaluation_rows: list[dict[str, Any]] = []
for run_name, predictions in PREDICTION_RUNS.items():
    evaluation_rows.extend(evaluate_run(run_name, predictions, ACTIVE_METRICS))

RESULTS_DF = pd.DataFrame(evaluation_rows)
if not RESULTS_DF.empty:
    RESULTS_DF = RESULTS_DF.sort_values(
        ["run", "ground_truth_found", "prediction_line"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

display(RESULTS_DF.head())


,run,prediction_path,prediction_line,prediction_id,ground_truth_found,question,pred_answer,prediction_time_taken,ground_truth_id,ground_truth_qtype,ground_truth_tags,ground_truth_answer,answer_token_precision,answer_token_recall,answer_token_f1,gt_entity_total,gt_entity_covered,gt_entity_coverage,missing_gt_entities,evaluation_error
0,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,1,gt_0,True,"\nHow many unique ""experiment execution"" are t...",There are 17 unique experiment executions reco...,78.313537,gt_0,"[""single"", ""numeric"", ""ISP""]",{},The answer to the question is 12 unique execut...,0.300000,0.333333,0.315789,1.0,0.0,0.0,"[""12""]",NaN
1,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,2,gt_1,True,\nIn what places do we utilize AI in this work...,AI is utilized in this workflow primarily thro...,51.730747,gt_1,"[""multi"", ""entity"", ""SI""]",{},\nThe ChatBS System utilizes LLMs to generate ...,0.288462,0.319149,0.303030,10.0,3.0,0.3,"[""ChatBS-NexGen:Generative_Task-id_20260420105...",NaN
2,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,3,gt_2,True,\nWhat are the extracted KG entities for the e...,The extracted Knowledge Graph (KG) entities ar...,100.572965,gt_2,"[""multi"", ""literal"", ""ISP""]",{},\nThe retrieved entities from the knowledge gr...,0.179487,0.175000,0.177215,6.0,0.0,0.0,"[""Chicken"", ""Turkey"", ""Yogurt"", ""Cheese"", ""Tun...",NaN
3,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,4,gt_3,True,\nwhy did some ingredients have missing values...,The Query Result Post-Processor is programmed ...,31.982723,gt_3,"[""multi"", ""literal"", ""ISP""]",{},\nDue to the fact that some information is not...,0.272727,0.281250,0.276923,4.0,0.0,0.0,"[""Chicken"", ""Turkey"", ""Cheese"", ""Tuna""]",NaN
4,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,5,gt_4,True,\nwhat are the instructions used by the LLM to...,"```json { ""answer"": ""The LLM used two sets of ...",27.911092,gt_4,"[""multi"", ""entity"", ""literal"", ""SI""]",{},\n We utilize the following input instructi...,0.395210,0.776471,0.523810,3.0,0.0,0.0,"[""http://testwebsite/testProgram#query_result_...",NaN


In [8]:
def build_run_summary(results_df: pd.DataFrame) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame()

    counts_df = (
        results_df.groupby("run", dropna=False)
        .agg(
            evaluated_examples=("run", "size"),
            matched_ground_truth=("ground_truth_found", "sum"),
            avg_prediction_time_taken=("prediction_time_taken", "mean"),
        )
        .reset_index()
    )

    numeric_columns = [
        column
        for column in results_df.select_dtypes(include="number").columns
        if column not in {"prediction_line", "prediction_time_taken"}
    ]

    if not numeric_columns:
        return counts_df

    summary_df = (
        results_df.groupby("run", dropna=False)[numeric_columns]
        .agg(["mean", "std"])
        .reset_index()
    )
    summary_df.columns = [
        "run" if column == ("run", "") else f"{column[0]}_{column[1]}"
        for column in summary_df.columns
    ]

    return counts_df.merge(summary_df, on="run", how="left")


RUN_SUMMARY_DF = build_run_summary(RESULTS_DF)
display(RUN_SUMMARY_DF)

UNMATCHED_PREDICTIONS_DF = RESULTS_DF.loc[~RESULTS_DF["ground_truth_found"]].copy()
display(UNMATCHED_PREDICTIONS_DF)

if SAVE_PER_EXAMPLE_CSV is not None:
    SAVE_PER_EXAMPLE_CSV.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_DF.to_csv(SAVE_PER_EXAMPLE_CSV, index=False)

if SAVE_SUMMARY_CSV is not None:
    SAVE_SUMMARY_CSV.parent.mkdir(parents=True, exist_ok=True)
    RUN_SUMMARY_DF.to_csv(SAVE_SUMMARY_CSV, index=False)


,run,evaluated_examples,matched_ground_truth,avg_prediction_time_taken,answer_token_precision_mean,answer_token_precision_std,answer_token_recall_mean,answer_token_recall_std,answer_token_f1_mean,answer_token_f1_std,gt_entity_total_mean,gt_entity_total_std,gt_entity_covered_mean,gt_entity_covered_std,gt_entity_coverage_mean,gt_entity_coverage_std
0,fullcontext_latest,112,103,41.007412,0.43314,0.172663,0.351598,0.179428,0.340337,0.153466,2.417476,2.881735,0.07767,0.362165,0.021002,0.113315


,run,prediction_path,prediction_line,prediction_id,ground_truth_found,question,pred_answer,prediction_time_taken,ground_truth_id,ground_truth_qtype,ground_truth_tags,ground_truth_answer,answer_token_precision,answer_token_recall,answer_token_f1,gt_entity_total,gt_entity_covered,gt_entity_coverage,missing_gt_entities,evaluation_error
103,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,104,gt_103,False,\nwhat is the number of output for the Output ...,"There is one generated collection, identified ...",30.227962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
104,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,105,gt_104,False,\nwhat is the number of output for the Output ...,The provided graph does not contain informatio...,27.910598,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
105,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,106,gt_105,False,\nwhat is the number of output for the Output ...,"There is one output, represented by a Collecti...",33.980603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
106,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,107,gt_106,False,\nwhat is the number of output for the Output ...,There is 1 output generated from the Output po...,22.465606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
107,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,108,gt_107,False,\nwhat is the number of output for the Output ...,The execution associated with http://testwebsi...,32.717301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
108,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,109,gt_108,False,\nwhat is the number of output for the Output ...,The execution with identifier http://testwebsi...,33.032730,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
109,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,110,gt_109,False,\nwhat is the number of output for the Output ...,There is one output connected from the Output ...,40.144253,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
110,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,111,gt_110,False,\nwhat is the number of output for the Output ...,There is evidence of at least one generated ou...,32.134354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found
111,fullcontext_latest,/home/desild/work/research/LLM-Workflow-Explor...,112,gt_111,False,\nwhat is the number of output for the LLM Cha...,There is 1 output associated with the LLM Chat...,27.590351,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ground_truth_not_found


In [10]:
sort_metric = None
for candidate in ("llm_accuracy", "answer_token_f1", "gt_entity_coverage"):
    if candidate in RESULTS_DF.columns:
        sort_metric = candidate
        break

if sort_metric is None:
    print("No sortable metric column found yet.")
else:
    inspection_columns = [
        column
        for column in [
            "run",
            "prediction_id",
            "question",
            sort_metric,
            "gt_entity_coverage",
            "pred_answer",
            "ground_truth_answer",
        ]
        if column in RESULTS_DF.columns
    ]
    display(RESULTS_DF.sort_values(sort_metric, ascending=True)[inspection_columns].head(10))


,run,prediction_id,question,answer_token_f1,gt_entity_coverage,pred_answer,ground_truth_answer
6,fullcontext_latest,gt_6,\nwhat are the instructions used by the LLM to...,0.079254,0.0000,The LLM used instructions to extract important...,\n We utilize the following input instructi...
50,fullcontext_latest,gt_50,\nWhat are the outputs of the Generative Task ...,0.082126,0.0000,The output of the Generative Task component wi...,\n The inputs to the LLM used for the Gener...
47,fullcontext_latest,gt_47,\nWhat are the inputs of the Generative Task t...,0.083538,0.0000,The Generative Task used for information extra...,\n The inputs to the LLM used for the Gener...
62,fullcontext_latest,gt_62,\nWhat Generative Task generated the data that...,0.087591,0.0000,The provided knowledge graph does not contain ...,\n The Generative Task that generated the d...
56,fullcontext_latest,gt_56,\nWhat Generative Task generated the data that...,0.088889,0.0000,The provided graph is insufficient to determin...,\n The Generative Task that generated the d...
65,fullcontext_latest,gt_65,\nWhat Generative Task generated the data that...,0.088889,0.0000,The provided graph is insufficient to determin...,\n The Generative Task that generated the d...
64,fullcontext_latest,gt_64,\nWhat Generative Task generated the data that...,0.091566,0.0000,The graph shows that the execution http://test...,\n The Generative Task that generated the d...
60,fullcontext_latest,gt_60,\nWhat Generative Task generated the data that...,0.092457,0.0000,The provided provenance knowledge graph does n...,\n The Generative Task that generated the d...
66,fullcontext_latest,gt_66,\nWhat Generative Task generated the data that...,0.093827,0.0000,The graph is insufficient to determine which s...,\n The Generative Task that generated the d...
40,fullcontext_latest,gt_40,\nDoes any of the executions of the sparql que...,0.094421,0.0625,"No, there are no triples in the provided knowl...",\n Executions of the sparql query function ...
